
# P3 — Chief Complaint Feature Engineering

**Project:** Predicting Opioid Dispensing in Low Back Pain Emergency Visits: Explainable Machine Learning and Clinical Phenotype Sensitivity

**Dataset:** MIMIC-IV-ED v2.2  
**Unit of Analysis:** ED encounter (`stay_id`)  
**Input:** Triage chief complaint text  
**Downstream Use:** P4 encounter-level integration and P6 predictive modeling

## 1. Study Context & Scope

### 1.1 Background

Chief complaint text provides a brief description of a patient's presenting concern at emergency department (ED) triage. Although often short, abbreviated, and inconsistently documented, it may capture clinically relevant information not fully represented by structured triage variables.

This study investigates whether chief complaint text contributes additional predictive information about documented opioid dispensing among ED encounters retrospectively identified using an ICD-defined low back pain (LBP) phenotype.

The analytical cohorts are defined separately in P2:

- **C1 — Strict LBP:** The primary cohort, identified using explicit LBP-related ICD codes.
- **C2 — Expanded LBP:** A broader sensitivity cohort that includes C1 and additional sciatica or back pain symptom codes.

Chief complaint text is treated exclusively as a potential **triage-time predictive feature**, not as an alternative method for establishing C1/C2 cohort eligibility.

### 1.2 Objectives

The objectives of P3 are to:

1. Assess the availability and basic quality of encounter-level chief complaint text.
2. Develop deterministic, reproducible text normalization procedures.
3. Extract a small set of clinically interpretable, rule-based text indicators.
4. Perform a focused audit of rule behavior, including ambiguous expressions and potential false matches.
5. Prepare normalized text and rule-based features for downstream encounter integration and predictive modeling.

The resulting features will be evaluated for incremental predictive value in P6 rather than interpreted as confirmed clinical diagnoses.

### 1.3 Methodological Approach

P3 uses lightweight, rule-based clinical concept extraction rather than developing a comprehensive natural language processing (NLP) classifier.

The workflow consists of:

1. **Input QA:** Evaluate encounter identifiers, missing text, and basic text characteristics.
2. **Text normalization:** Standardize case, whitespace, and selected punctuation or abbreviations while preserving relevant clinical expressions.
3. **Clinical concept extraction:** Generate non-mutually-exclusive binary indicators for selected LBP-related concepts.
4. **Rule audit:** Apply regression tests and review a limited set of ambiguous or potentially incorrect matches.
5. **Feature export:** Produce an encounter-level feature dataset for integration with the ICD-defined cohorts.

A complementary statistical NLP representation using TF-IDF, with optional TruncatedSVD, will be evaluated in P6. Its vocabulary, IDF weights, and any dimensionality-reduction components will be fitted using training data only, after patient-level partitioning.

### 1.4 Scope and Limitations

**In scope**

- Encounter-level chief complaint data quality checks.
- Lightweight text normalization.
- Prespecified clinical keyword and phrase matching.
- Independent, potentially overlapping clinical concept indicators.
- Focused rule regression tests and error review.
- Reusable feature definitions and documented downstream interfaces.

**Out of scope**

- Chief complaint-based cohort identification or modification of C1/C2 eligibility.
- Independent clinical phenotype validation.
- Large-scale manual or AI-assisted annotation.
- Transformer fine-tuning or comprehensive clinical NLP.
- TF-IDF/SVD model fitting before the P6 train/test split.
- Optimization of clinical text rules using final diagnoses or opioid dispensing outcomes.

### 1.5 Expected Deliverables

P3 will produce:

- A documented chief complaint normalization procedure.
- A compact set of clinically interpretable binary text features.
- Focused rule QA and regression-test results.
- An encounter-level feature dataset keyed by `stay_id`.
- A reproducible feature specification for P4 and P6.

The final features will be interpreted as operational representations of triage language rather than clinically validated diagnostic classifications.



## 2. Input Inventory & Text Normalization

### 2.1 Input Data Loading & Integrity QA

**Objective**

Load the encounter-level chief complaint dataset and verify its structural integrity and text availability before feature engineering.

**Data Source**

- Source table: `physionet-data.mimiciv_ed.triage`
- Extraction SQL: `sql/03_chief_complaint/01_extract_chief_complaint.sql`
- Local input: `results/03_chief_complaint/chiefcomplaint_encounter_raw.csv`
- Unit of analysis: ED encounter (`stay_id`)

The SQL extraction preserves the original chief complaint text without filtering by ICD diagnosis, clinical keywords, or opioid dispensing outcomes.

**Method**

1. Load encounter identifiers and chief complaint text while preserving their original string representations.
2. Verify the expected columns and uniqueness of `stay_id`.
3. Identify missing or blank chief complaint values.
4. Identify common placeholder expressions representing unavailable text.
5. Summarize data completeness and determine whether the dataset is suitable for deterministic text normalization.

The analysis reuses the existing extraction rather than repeating previously completed exploratory SQL queries.

No cohort filtering or clinical text classification is performed at this stage.


In [1]:

# Global setup
from pathlib import Path
import re
import pandas as pd

# Notebook working directory: project_root/notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

RESULTS_DIR = PROJECT_ROOT / "results" / "03_chief_complaint"
INPUT_PATH = RESULTS_DIR / "chiefcomplaint_encounter_raw.csv"

assert INPUT_PATH.is_file(), f"Input file not found: {INPUT_PATH}"

print("Project root:", PROJECT_ROOT)
print("Input file:", INPUT_PATH)


Project root: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction
Input file: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/03_chief_complaint/chiefcomplaint_encounter_raw.csv


In [2]:

# 2.1 Load encounter-level chief complaint data

df_cc_raw = pd.read_csv(
    INPUT_PATH,
    dtype={
        "stay_id": "string",
        "chiefcomplaint": "string"
    },
    keep_default_na=False
)

# Validate schema and encounter identifiers
required_columns = {"stay_id", "chiefcomplaint"}

assert required_columns.issubset(df_cc_raw.columns), (
    f"Missing columns: {required_columns - set(df_cc_raw.columns)}"
)

total_rows = len(df_cc_raw)
unique_stays = df_cc_raw["stay_id"].nunique()

missing_stay_ids = (
    df_cc_raw["stay_id"].str.strip().eq("").sum()
)

duplicate_stays = df_cc_raw["stay_id"].duplicated().sum()

# Identify empty, blank, and placeholder complaints
complaint_clean = (
    df_cc_raw["chiefcomplaint"]
    .str.strip()
    .str.lower()
)

PLACEHOLDERS = {
    "n/a", "na", "none", "null", "unknown",
    "not available", "not documented"
}

blank_complaints = complaint_clean.eq("")
placeholder_complaints = complaint_clean.isin(PLACEHOLDERS)

valid_complaints = ~(
    blank_complaints | placeholder_complaints
)

qa_summary = pd.DataFrame({
    "metric": [
        "Total rows",
        "Unique ED encounters",
        "Missing stay_id",
        "Duplicate stay_id",
        "Blank chief complaints",
        "Placeholder complaints",
        "Usable chief complaints",
        "Usable text (%)"
    ],
    "value": [
        total_rows,
        unique_stays,
        missing_stay_ids,
        duplicate_stays,
        int(blank_complaints.sum()),
        int(placeholder_complaints.sum()),
        int(valid_complaints.sum()),
        round(valid_complaints.mean() * 100, 4)
    ]
})

display(qa_summary)

# Structural QA: fail explicitly if the input is not encounter-level
assert missing_stay_ids == 0, "Missing stay_id detected"
assert duplicate_stays == 0, "Duplicate stay_id detected"

print("Encounter-level integrity QA passed.")


,metric,value
0,Total rows,425087.0000
1,Unique ED encounters,425087.0000
2,Missing stay_id,0.0000
3,Duplicate stay_id,0.0000
4,Blank chief complaints,22.0000
5,Placeholder complaints,9.0000
6,Usable chief complaints,425056.0000
7,Usable text (%),99.9927


Encounter-level integrity QA passed.



#### Interpretation

The source dataset contains 425,087 ED encounters, with one unique `stay_id` per record and no missing encounter identifiers or duplicate records.

Initial raw-text screening identified 22 blank complaints and 9 predefined placeholder expressions. The remaining 425,056 records (99.9927%) contained nonblank, non-placeholder raw text.

This represents preliminary text availability rather than final normalized-text usability. Additional records may become empty after punctuation and placeholder-symbol normalization, which is assessed in Section 2.2.

All encounters will be retained regardless of text availability. No SQL re-extraction, deduplication, or ICD-based filtering is required.



### 2.2 Text Normalization

**Objective**

Develop a deterministic normalization procedure that standardizes common formatting variations in short ED chief complaint text while preserving clinically meaningful expressions.

**Method**

The normalization procedure will:

1. Convert text to lowercase.
2. Normalize selected punctuation and separators to spaces.
3. Collapse repeated whitespace.
4. Convert blank and predefined placeholder expressions to an empty string.
5. Preserve clinically meaningful words, abbreviations, negations, and anatomical terms.

No stemming, lemmatization, stop-word removal, or model-based semantic interpretation is performed.

Both the original and normalized chief complaint fields will be retained to support auditing and downstream TF-IDF feature extraction.


In [3]:

# 2.2 Deterministic chief complaint normalization

def normalize_chief_complaint(text):
    """Normalize formatting without clinical classification."""
    if pd.isna(text):
        return ""

    text = str(text).strip().lower()

    if text in PLACEHOLDERS:
        return ""

    # Replace common separators with spaces.
    # Preserve words and their original sequence.
    text = re.sub(r"[/,;|]+", " ", text)

    # Normalize remaining punctuation while retaining letters/digits.
    text = re.sub(r"[^\w\s_]+|_+", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


df_cc = df_cc_raw.copy()

df_cc["cc_text_normalized"] = (
    df_cc["chiefcomplaint"].map(normalize_chief_complaint)
    .astype("string")
)

df_cc["cc_text_missing"] = (
    df_cc["cc_text_normalized"].eq("").astype("int8")
)

# Basic normalization checks

assert len(df_cc) == len(df_cc_raw)
assert df_cc["stay_id"].is_unique
assert df_cc["cc_text_normalized"].notna().all()

# Previously missing complaints must remain missing
raw_unavailable = blank_complaints | placeholder_complaints

assert df_cc.loc[
    raw_unavailable, "cc_text_missing"
].eq(1).all()

# Check missingness after normalization
normalized_missing = int(df_cc["cc_text_missing"].sum())
newly_empty_count = int(
    (~raw_unavailable & df_cc["cc_text_missing"].eq(1)).sum()
)

print("Total encounters:", len(df_cc))
print("Raw unavailable text:", int(raw_unavailable.sum()))
print("Newly empty after normalization:", newly_empty_count)
print("Total normalized missing text:", normalized_missing)
print(
    "Usable normalized text (%):",
    round((1 - normalized_missing / len(df_cc)) * 100, 4)
)


Total encounters: 425087
Raw unavailable text: 31
Newly empty after normalization: 1301
Total normalized missing text: 1332
Usable normalized text (%): 99.6867



#### Interpretation

Deterministic normalization was applied to all 425,087 chief complaint records without excluding any ED encounters.

Initial raw-text screening identified 31 unavailable complaints (22 blank values and 9 predefined placeholders). An additional 1,301 records became empty after punctuation, separator, and underscore normalization.

Consequently, 1,332 encounters (0.3133%) have empty normalized chief complaint text, leaving 423,755 encounters (99.6867%) with nonempty normalized text.

These records are retained with `cc_text_missing = 1`. This indicator represents the absence of text after the specified normalization procedure, rather than independently verified clinical information availability.

The normalization procedure preserves the remaining text for subsequent feature engineering and downstream TF-IDF representation. This assessment concerns formatting and text availability, not clinical feature accuracy.



### 2.3 Normalization Quality Assessment

**Objective**

Assess whether deterministic text normalization preserves clinically relevant information and produces consistent outputs for common ED chief complaint expressions.

**Method**

A small set of representative expressions will be used to evaluate:

1. Common abbreviations and anatomical terminology.
2. Multi-site complaints and punctuation-separated phrases.
3. Negation expressions.
4. Laterality and radiating pain descriptions.
5. Blank, placeholder, and symbol-only inputs.

This assessment focuses on formatting consistency and information preservation. It does not evaluate the diagnostic accuracy of clinical concept extraction.

Expressions with unresolved clinical ambiguity will be documented for the rule-based feature engineering stage rather than interpreted during normalization.


In [4]:

# 2.3 Targeted normalization tests

normalization_cases = [
    ("LOW BACK PAIN", "low back pain", "Case normalization"),
    ("  Lower   Back  Pain ", "lower back pain", "Whitespace"),
    ("LBP", "lbp", "Clinical abbreviation"),
    ("SCIATICA", "sciatica", "Clinical terminology"),
    ("BACK/LEG PAIN", "back leg pain", "Multi-site complaint"),
    ("BACK/THIGH PAIN", "back thigh pain", "Multi-site complaint"),
    ("L/S PAIN", "l s pain", "Anatomical abbreviation"),
    ("NO BACK PAIN", "no back pain", "Negation"),
    ("LOW BACK PAIN RADIATING TO LEFT LEG",
     "low back pain radiating to left leg", "Radiation and laterality"),
    ("BACK PAIN, FALL", "back pain fall", "Punctuation"),
    ("R-SIDED BACK PAIN", "r sided back pain", "Laterality"),
    ("N/A", "", "Placeholder"),
    ("--", "", "Symbol-only text"),
    ("___", "", "Underscore-only placeholder"),
    ("", "", "Blank input"),
    (None, "", "Null input"),
]

normalization_qa = pd.DataFrame(
    [
        {
            "raw_text": raw,
            "expected": expected,
            "actual": normalize_chief_complaint(raw),
            "test_purpose": purpose
        }
        for raw, expected, purpose in normalization_cases
    ]
)

normalization_qa["passed"] = (
    normalization_qa["actual"] == normalization_qa["expected"]
)

display(normalization_qa)

assert normalization_qa["passed"].all(), (
    "Normalization test failure detected."
)

print(
    f"Normalization tests passed: "
    f"{normalization_qa['passed'].sum()}/{len(normalization_qa)}"
)


,raw_text,expected,actual,test_purpose,passed
0,LOW BACK PAIN,low back pain,low back pain,Case normalization,True
1,Lower Back Pain,lower back pain,lower back pain,Whitespace,True
2,LBP,lbp,lbp,Clinical abbreviation,True
3,SCIATICA,sciatica,sciatica,Clinical terminology,True
4,BACK/LEG PAIN,back leg pain,back leg pain,Multi-site complaint,True
5,BACK/THIGH PAIN,back thigh pain,back thigh pain,Multi-site complaint,True
6,L/S PAIN,l s pain,l s pain,Anatomical abbreviation,True
7,NO BACK PAIN,no back pain,no back pain,Negation,True
8,LOW BACK PAIN RADIATING TO LEFT LEG,low back pain radiating to left leg,low back pain radiating to left leg,Radiation and laterality,True
9,"BACK PAIN, FALL",back pain fall,back pain fall,Punctuation,True


Normalization tests passed: 16/16


In [5]:

# 2.3 Full-dataset normalization consistency

# Normalization should be idempotent
normalized_twice = df_cc["cc_text_normalized"].map(
    normalize_chief_complaint
)

idempotent = (
    normalized_twice
    == df_cc["cc_text_normalized"]
).all()

# No leading/trailing or repeated whitespace
whitespace_valid = ~df_cc["cc_text_normalized"].str.contains(
    r"^\s|\s$|\s{2,}",
    regex=True,
    na=False
)

# Missing indicator must match empty normalized text
missing_consistent = (
    df_cc["cc_text_missing"].eq(1)
    == df_cc["cc_text_normalized"].eq("")
).all()

assert idempotent, "Normalization is not idempotent."
assert whitespace_valid.all(), "Whitespace inconsistency found."
assert missing_consistent, "Missing-text indicator mismatch."

print("Idempotence check: PASS")
print("Whitespace consistency: PASS")
print("Missing indicator consistency: PASS")
print("Total normalized missing:", int(df_cc["cc_text_missing"].sum()))


Idempotence check: PASS
Whitespace consistency: PASS
Missing indicator consistency: PASS
Total normalized missing: 1332


In [6]:

# Focused manual spot check: reproducible sample
sample_review = df_cc.loc[
    df_cc["cc_text_missing"].eq(0),
    ["chiefcomplaint", "cc_text_normalized"]
].sample(n=15, random_state=42)

display(sample_review)


,chiefcomplaint,cc_text_normalized
256510,CP,cp
313978,Hypertension,hypertension
27863,Lower back pain,lower back pain
251294,L Arm pain,l arm pain
63341,"s/p Fall, Back pain, R Arm injury",s p fall back pain r arm injury
193256,ABD PAIN,abd pain
44256,"s/p Fall, Weakness",s p fall weakness
259192,Chest pain,chest pain
277474,"AH, HI, SI",ah hi si
48226,"N/V, L ANKLE PAIN",n v l ankle pain



#### Interpretation

The targeted normalization tests confirmed consistent handling of case, whitespace, punctuation, common clinical abbreviations, negation, and unavailable text.

Full-dataset quality checks confirmed that normalization is idempotent, whitespace is consistently standardized, and the missing-text indicator agrees with the normalized text field.

Certain expressions remain clinically ambiguous after normalization. For example, punctuation-separated complaints such as "BACK/LEG PAIN" preserve their component words but do not establish a radiating pain relationship.

These ambiguities will be addressed conservatively during clinical rule development. No diagnostic inference is performed during text normalization.

The normalized chief complaint field is ready for lightweight rule-based feature extraction in Section 3.



## 3. Lightweight Clinical Rule Features

### 3.1 Clinical Concept Specification

**Objective**

Define a small, interpretable, and reproducible set of clinical text indicators derived from ED chief complaint descriptions for downstream opioid dispensing prediction.

Unlike the ICD-based phenotypes developed in P2, these indicators are not used to establish cohort eligibility. They represent observable language in triage documentation rather than confirmed clinical conditions.

**Prespecified Clinical Features**

| Feature | Positive Textual Evidence | Boundary / Exclusion |
|---|---|---|
| `cc_explicit_lbp` | Explicit low or lower back pain, lumbar pain, lumbago, or recognized LBP terminology | Does not infer lumbar pain from anatomically unspecified back pain |
| `cc_sciatica` | Explicit, affirmed sciatica or sciatic pain terminology | Does not infer sciatica from isolated leg pain, numbness, or radiculopathy terminology |
| `cc_broad_back_pain` | Any affirmed back-pain-related expression, including those identified by `cc_explicit_lbp` | Defined as a superset of `cc_explicit_lbp`; does not include explicitly negated mentions |
| `cc_radiating_leg_pain` | Explicit descriptions of pain radiating or shooting from the back toward the leg, thigh, or lower extremity | Does not infer radiation merely from co-occurring back and leg pain or isolated radiculopathy terminology |

All four features are binary and non-mutually-exclusive. Multiple indicators may be positive for the same encounter.

An indicator value of `0` means that the corresponding textual evidence was not detected. It does not establish the clinical absence of the condition.

**Feature Hierarchy**

`cc_broad_back_pain` is defined as a logical superset of `cc_explicit_lbp`.

For every encounter:

- If `cc_explicit_lbp = 1`, then `cc_broad_back_pain = 1`.
- If `cc_broad_back_pain = 1`, `cc_explicit_lbp` may be either `0` or `1`.

This hierarchy ensures that recognized abbreviations such as `LBP` and clinical terminology such as `LUMBAGO` are consistently represented as back-pain-related mentions, even when the literal phrase "back pain" is absent.

The hierarchy will be enforced and checked programmatically.

**Rule Design Principles**

1. **Explicit textual evidence:** Rules prioritize directly documented expressions rather than inferred diagnoses or symptoms.
2. **Conservative assertion handling:** Selected explicit negation and uncertainty expressions will be handled to reduce obvious false-positive matches.
3. **Independent indicators:** Features may overlap, and no mutually exclusive clinical phenotype hierarchy is imposed beyond the defined broad-back-pain inclusion relationship.
4. **Deterministic implementation:** Identical input text must produce identical feature values.
5. **Outcome independence:** Rules will not be optimized using final ICD diagnoses, opioid dispensing outcomes, or downstream model performance.
6. **Limited complexity:** New patterns will be introduced only to address clear, recurring errors supported by targeted tests or focused review.

**Assertion and Ambiguity Handling**

Chief complaints may contain multiple symptom expressions, negated statements, or uncertain diagnostic terminology. The following conservative conventions will be applied:

- **Local negation:** Explicit negation expressions, such as "no back pain" or "denies sciatica," will suppress the affected concept within a limited local clause or phrase.
- **Clause boundaries:** Negation will not automatically suppress unrelated affirmed symptoms elsewhere in the same chief complaint. Original punctuation and clause boundaries may be consulted during rule evaluation.
- **Diagnostic uncertainty:** Expressions such as "R/O sciatica" or "sciatica vs lumbar strain" will not be treated as affirmed evidence of the uncertain clinical concept.
- **Conflicting mentions:** When the same concept appears in both affirmed and negated contexts, rules will evaluate the mentions locally rather than automatically assigning a negative value to the entire complaint.
- **Ambiguous multi-site complaints:** Expressions such as "BACK/LEG PAIN" will not be interpreted as radiating pain without explicit directional or radiation terminology.
- **Abbreviations:** Common, clearly interpretable abbreviations may be recognized; ambiguous shorthand will not be expanded into inferred diagnoses without sufficient textual evidence.

These conventions are intended to handle common and clearly identifiable cases. They do not constitute comprehensive clinical negation, uncertainty, or assertion detection.

**Illustrative Examples**

| Chief Complaint | Explicit LBP | Sciatica | Broad Back Pain | Radiating Leg Pain |
|---|---:|---:|---:|---:|
| `LOW BACK PAIN` | 1 | 0 | 1 | 0 |
| `LBP` | 1 | 0 | 1 | 0 |
| `LUMBAGO` | 1 | 0 | 1 | 0 |
| `BACK PAIN` | 0 | 0 | 1 | 0 |
| `SCIATICA` | 0 | 1 | 0 | 0 |
| `LOW BACK PAIN RADIATING TO LEFT LEG` | 1 | 0 | 1 | 1 |
| `BACK/LEG PAIN` | 0 | 0 | 1 | 0 |
| `NO BACK PAIN` | 0 | 0 | 0 | 0 |
| `BACK PAIN, NO LEG PAIN` | 0 | 0 | 1 | 0 |
| `NO BACK PAIN, SCIATICA` | 0 | 1 | 0 | 0 |
| `LBP, R/O SCIATICA` | 1 | 0 | 1 | 0 |
| `SCIATICA VS LUMBAR STRAIN` | 0 | 0 | 0 | 0 |
| `LBP W/ RADICULOPATHY` | 1 | 0 | 1 | 0 |

The examples represent prespecified expected rule behavior, not independently validated clinical annotations. They will inform regression tests during implementation.

**Implementation and Validation Boundary**

P3 will implement these prespecified concepts using lightweight, deterministic text-processing rules.

The implementation will:

1. Reuse the normalization approach developed in Section 2, while consulting original text where punctuation or clause boundaries are needed.
2. Produce four independent binary clinical indicators.
3. Enforce the `cc_explicit_lbp` → `cc_broad_back_pain` inclusion relationship.
4. Apply limited, documented negation and uncertainty handling.
5. Evaluate feature coverage, overlap, and expected behavior using targeted regression tests.
6. Conduct a focused audit of representative matches and potential false positives.

The rules will be frozen after the focused quality assessment. They will not be repeatedly expanded to maximize keyword coverage or predictive performance.

These features are operational text indicators rather than clinically validated phenotypes. Their incremental predictive value will be evaluated in P6.

TF-IDF and optional TruncatedSVD will be fitted separately in P6 after patient-level data partitioning to prevent training/test information leakage.



### 3.2 Rule Implementation

**Objective**

Implement the four prespecified clinical text indicators as deterministic, non-mutually-exclusive binary features.

**Method**

The implementation adapts lexical patterns from the exploratory P3 V2 workflow while removing mutually exclusive phenotype classification and category-priority rules.

Each clinical concept is identified using explicit textual evidence. Limited local negation and diagnostic uncertainty handling is applied to reduce obvious false-positive matches.

Original chief complaint text is retained for rule evaluation because the normalization performed in Section 2 removes punctuation and may obscure clause boundaries or abbreviations such as "R/O".

The implementation enforces the following feature hierarchy:

`cc_explicit_lbp = 1` implies `cc_broad_back_pain = 1`.

**Implementation Source:** `src/chief_complaint_rules.py`

The rule set is intended for feature engineering, not clinical diagnosis or independent phenotype validation. Its behavior will be checked through targeted examples and regression tests before being frozen.


In [7]:

# 3.2 Apply reusable clinical concept rules

import sys

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from chief_complaint_rules import (
    extract_cc_features,
    add_cc_features,
)

# Apply rules to original chief complaint text.
# Keep normalized text and missingness fields from Section 2.
df_cc = add_cc_features(df_cc)

CLINICAL_FEATURES = [
    "cc_explicit_lbp",
    "cc_sciatica",
    "cc_broad_back_pain",
    "cc_radiating_leg_pain",
]

# Structural and logical integrity checks
assert len(df_cc) == len(df_cc_raw)
assert df_cc["stay_id"].is_unique

assert not (
    (df_cc["cc_explicit_lbp"] == 1)
    & (df_cc["cc_broad_back_pain"] == 0)
).any()

assert (
    df_cc.loc[
        df_cc["cc_text_missing"] == 1,
        CLINICAL_FEATURES
    ].sum().sum() == 0
)

print("Clinical feature extraction complete.")
print("Total encounters:", len(df_cc))

display(
    df_cc[
        ["chiefcomplaint", *CLINICAL_FEATURES]
    ].head(15)
)


Clinical feature extraction complete.
Total encounters: 425087


,chiefcomplaint,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain
0,CHANGE IN MENTAL STATUS,0,0,0,0
1,"ETOH, Unable to ambulate",0,0,0,0
2,Cough,0,0,0,0
3,s/p Fall,0,0,0,0
4,L Ear pain,0,0,0,0
5,N,0,0,0,0
6,Abnormal labs,0,0,0,0
7,"Abd pain, Vomiting",0,0,0,0
8,ALLERGIC REACTION,0,0,0,0
9,S/P ASSAULT,0,0,0,0


In [8]:

# 3.2 Minimal implementation sanity checks

quick_cases = [
    "LOW BACK PAIN",
    "LBP",
    "SCIATICA",
    "BACK PAIN",
    "LOW BACK PAIN RADIATING TO LEFT LEG",
    "BACK/LEG PAIN",
    "NO BACK PAIN",
    "BACK PAIN, NO LEG PAIN",
    "NO BACK PAIN, SCIATICA",
    "LBP, R/O SCIATICA",
    "SCIATICA VS LUMBAR STRAIN",
]

quick_results = pd.DataFrame([
    {"complaint": text, **extract_cc_features(text)}
    for text in quick_cases
])

display(quick_results)

assert quick_results.loc[
    quick_results["complaint"] == "LBP",
    "cc_explicit_lbp"
].iloc[0] == 1

assert quick_results.loc[
    quick_results["complaint"] == "NO BACK PAIN",
    "cc_broad_back_pain"
].iloc[0] == 0

assert quick_results.loc[
    quick_results["complaint"] == "LBP, R/O SCIATICA",
    "cc_sciatica"
].iloc[0] == 0

print("Initial sanity checks passed.")


,complaint,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain
0,LOW BACK PAIN,1,0,1,0
1,LBP,1,0,1,0
2,SCIATICA,0,1,0,0
3,BACK PAIN,0,0,1,0
4,LOW BACK PAIN RADIATING TO LEFT LEG,1,0,1,1
5,BACK/LEG PAIN,0,0,1,0
6,NO BACK PAIN,0,0,0,0
7,"BACK PAIN, NO LEG PAIN",0,0,1,0
8,"NO BACK PAIN, SCIATICA",0,1,0,0
9,"LBP, R/O SCIATICA",1,0,1,0


Initial sanity checks passed.



#### Interpretation

The prespecified clinical concept definitions were implemented as four independent binary indicators using a reusable Python rule module.

The implementation preserves encounter-level records, separates explicit LBP terminology from sciatica and radiating leg pain expressions, and enforces the intended inclusion relationship between explicit LBP and broad back pain.

Limited local negation and uncertainty handling is incorporated to reduce obvious false-positive matches. Original text is used during rule evaluation to preserve selected clause boundaries and clinical abbreviations.

These checks establish basic implementation consistency, not clinical validity. Subsequent sections will examine feature coverage, overlap, and rule behavior using targeted regression tests and focused audit samples.



### 3.3 Feature Coverage & Overlap

**Objective**

Summarize the coverage and overlap of the four prespecified clinical text indicators across ED chief complaint records.

**Method**

1. Calculate the number and percentage of encounters matching each indicator.
2. Quantify encounters with at least one clinical indicator.
3. Examine the distribution of the number of positive indicators per encounter.
4. Verify the expected feature hierarchy and summarize pairwise overlap.
5. Inspect a small set of common feature combinations.

This analysis uses the complete, unfiltered chief complaint extraction. It does not represent the ICD-defined C1/C2 study cohorts.

Feature prevalence is descriptive and will not be used to optimize rule coverage or downstream predictive performance.


In [9]:

# 3.3.1 Clinical feature coverage

n_encounters = len(df_cc)

feature_coverage = pd.DataFrame({
    "feature": CLINICAL_FEATURES,
    "n_positive": [
        int(df_cc[col].sum())
        for col in CLINICAL_FEATURES
    ]
})

feature_coverage["pct_encounters"] = (
    feature_coverage["n_positive"] / n_encounters * 100
).round(3)

display(feature_coverage)

# Count encounters matching at least one clinical indicator
df_cc["cc_any_clinical_feature"] = (
    df_cc[CLINICAL_FEATURES].any(axis=1).astype("int8")
)

n_any = int(df_cc["cc_any_clinical_feature"].sum())

print(f"Total encounters: {n_encounters:,}")
print(f"At least one clinical indicator: {n_any:,}")
print(f"Any indicator coverage: {n_any / n_encounters:.3%}")


,feature,n_positive,pct_encounters
0,cc_explicit_lbp,6033,1.419
1,cc_sciatica,20,0.005
2,cc_broad_back_pain,17101,4.023
3,cc_radiating_leg_pain,4,0.001


Total encounters: 425,087
At least one clinical indicator: 17,120
Any indicator coverage: 4.027%


In [10]:

# 3.3.2 Feature overlap and hierarchy QA

df_cc["cc_n_positive_features"] = (
    df_cc[CLINICAL_FEATURES].sum(axis=1).astype("int8")
)

feature_count_distribution = (
    df_cc["cc_n_positive_features"]
    .value_counts()
    .sort_index()
    .rename_axis("n_positive_features")
    .reset_index(name="n_encounters")
)

feature_count_distribution["pct_encounters"] = (
    feature_count_distribution["n_encounters"]
    / len(df_cc) * 100
).round(3)

display(feature_count_distribution)

# Pairwise overlap matrix (joint positive counts)
overlap_matrix = pd.DataFrame(
    {
        col: [
            int(((df_cc[row] == 1) & (df_cc[col] == 1)).sum())
            for row in CLINICAL_FEATURES
        ]
        for col in CLINICAL_FEATURES
    },
    index=CLINICAL_FEATURES
)

display(overlap_matrix)

# Prespecified logical constraints
hierarchy_violations = int(
    (
        (df_cc["cc_explicit_lbp"] == 1)
        & (df_cc["cc_broad_back_pain"] == 0)
    ).sum()
)

assert hierarchy_violations == 0
assert df_cc["cc_n_positive_features"].between(0, 4).all()
assert (
    df_cc["cc_n_positive_features"].gt(0)
    == df_cc["cc_any_clinical_feature"].eq(1)
).all()

print("Feature hierarchy and overlap QA: PASS")


,n_positive_features,n_encounters,pct_encounters
0,0,407967,95.973
1,1,11084,2.607
2,2,6034,1.419
3,3,2,0.000


,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain
cc_explicit_lbp,6033,0,6033,2
cc_sciatica,0,20,1,0
cc_broad_back_pain,6033,1,17101,4
cc_radiating_leg_pain,2,0,4,4


Feature hierarchy and overlap QA: PASS


In [11]:

# 3.3.3 Most common clinical feature combinations

feature_combinations = (
    df_cc.groupby(CLINICAL_FEATURES, dropna=False)
    .size()
    .reset_index(name="n_encounters")
    .sort_values("n_encounters", ascending=False)
)

feature_combinations["pct_encounters"] = (
    feature_combinations["n_encounters"]
    / len(df_cc) * 100
).round(3)

feature_combinations = feature_combinations.reset_index(drop=True)
display(feature_combinations)

print(
    "Distinct feature combinations:",
    len(feature_combinations)
)


,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain,n_encounters,pct_encounters
0,0,0,0,0,407967,95.973
1,0,0,1,0,11065,2.603
2,1,0,1,0,6031,1.419
3,0,1,0,0,19,0.004
4,0,0,1,1,2,0.000
5,1,0,1,1,2,0.000
6,0,1,1,0,1,0.000


Distinct feature combinations: 7



#### Interpretation

The four prespecified clinical text indicators were applied to 425,087 ED encounters without ICD-based cohort filtering.

At least one clinical indicator was identified in 17,120 encounters (4.027%). Broad back pain expressions were the most frequently detected (17,101; 4.023%), followed by explicit LBP expressions (6,033; 1.419%). Explicit sciatica terminology and radiating leg pain expressions were identified in 20 and 4 encounters, respectively.

The feature hierarchy was satisfied: all 6,033 encounters with an explicit LBP indicator also had a positive broad back pain indicator. Seven distinct feature combinations were observed, consistent with the non-mutually-exclusive feature design.

Most encounters (95.973%) did not match any prespecified concept. This is expected because the analysis includes the complete ED population rather than an ICD-defined LBP cohort.

The low detection frequencies for explicit sciatica and radiating leg pain remain a limitation requiring focused review. Feature prevalence alone does not establish clinical accuracy or justify expanding the rule vocabulary.




### 3.4 Targeted Regression Testing

**Objective**

Evaluate whether the implemented clinical text rules behave consistently with the prespecified feature definitions and assertion-handling conventions established in Section 3.1.

**Method**

A targeted regression-test suite will cover:

1. Explicit LBP, sciatica, and broad back pain expressions.
2. Clear descriptions of pain radiating toward the lower extremity.
3. Feature overlap and the explicit-LBP-to-broad-back-pain hierarchy.
4. Local negation and uncertain diagnostic expressions.
5. Ambiguous multi-site complaints and clinical abbreviations.
6. Missing and non-informative text.

Expected feature values are specified independently of the rule implementation. All four indicators are evaluated for each test case.

Failed cases will be examined against the original clinical concept definitions. Rules will be corrected only when failures demonstrate a clear inconsistency with the prespecified behavior.

These tests evaluate implementation consistency and selected semantic boundaries. They do not establish clinical sensitivity, specificity, or external validity.


In [12]:

# 3.4.1 Prespecified clinical rule regression tests

# Feature order:
# [explicit_lbp, sciatica, broad_back_pain, radiating_leg_pain]

test_cases = [
    # A. Prespecified Section 3.1 examples
    ("LOW BACK PAIN",                  [1, 0, 1, 0]),
    ("LBP",                            [1, 0, 1, 0]),
    ("LUMBAGO",                        [1, 0, 1, 0]),
    ("BACK PAIN",                      [0, 0, 1, 0]),
    ("SCIATICA",                       [0, 1, 0, 0]),
    ("LOW BACK PAIN RADIATING TO LEFT LEG",
                                       [1, 0, 1, 1]),
    ("BACK/LEG PAIN",                  [0, 0, 1, 0]),
    ("NO BACK PAIN",                   [0, 0, 0, 0]),
    ("BACK PAIN, NO LEG PAIN",         [0, 0, 1, 0]),
    ("NO BACK PAIN, SCIATICA",         [0, 1, 0, 0]),
    ("LBP, R/O SCIATICA",              [1, 0, 1, 0]),
    ("SCIATICA VS LUMBAR STRAIN",      [0, 0, 0, 0]),
    ("LBP W/ RADICULOPATHY",          [1, 0, 1, 0]),

    # B. Additional boundary cases
    ("LBP RAD TO RLE",                 [1, 0, 1, 1]),
    ("LBP SHOOTING DOWN LEG",          [1, 0, 1, 1]),
    ("LEG PAIN",                       [0, 0, 0, 0]),
    ("UPPER BACK PAIN",                [0, 0, 1, 0]),
    ("DENIES SCIATICA, BACK PAIN",     [0, 0, 1, 0]),
    ("NO BACK PAIN RADIATING TO LEG",  [0, 0, 0, 0]),
    ("R/O SCIATICA",                   [0, 0, 0, 0]),
    ("N/A",                            [0, 0, 0, 0]),
    ("",                               [0, 0, 0, 0]),
]

test_results = []

for text, expected in test_cases:
    actual_dict = extract_cc_features(text)
    actual = [actual_dict[f] for f in CLINICAL_FEATURES]

    test_results.append({
        "chief_complaint": text,
        "expected": expected,
        "actual": actual,
        "passed": actual == expected
    })

regression_qa = pd.DataFrame(test_results)

display(regression_qa)

n_passed = int(regression_qa["passed"].sum())
n_total = len(regression_qa)

print(f"Regression tests passed: {n_passed}/{n_total}")

# Display failures separately for diagnosis
failed_tests = regression_qa.loc[~regression_qa["passed"]]

if not failed_tests.empty:
    print("Tests requiring review:")
    display(failed_tests)
else:
    print("All targeted regression tests passed.")


,chief_complaint,expected,actual,passed
0,LOW BACK PAIN,"[1, 0, 1, 0]","[1, 0, 1, 0]",True
1,LBP,"[1, 0, 1, 0]","[1, 0, 1, 0]",True
2,LUMBAGO,"[1, 0, 1, 0]","[1, 0, 1, 0]",True
3,BACK PAIN,"[0, 0, 1, 0]","[0, 0, 1, 0]",True
4,SCIATICA,"[0, 1, 0, 0]","[0, 1, 0, 0]",True
5,LOW BACK PAIN RADIATING TO LEFT LEG,"[1, 0, 1, 1]","[1, 0, 1, 1]",True
6,BACK/LEG PAIN,"[0, 0, 1, 0]","[0, 0, 1, 0]",True
7,NO BACK PAIN,"[0, 0, 0, 0]","[0, 0, 0, 0]",True
8,"BACK PAIN, NO LEG PAIN","[0, 0, 1, 0]","[0, 0, 1, 0]",True
9,"NO BACK PAIN, SCIATICA","[0, 1, 0, 0]","[0, 1, 0, 0]",True


Regression tests passed: 22/22
All targeted regression tests passed.




#### Interpretation

All 22 prespecified regression tests passed following a targeted correction to the broad back pain rule for multi-site expressions such as "BACK/LEG PAIN".

The tests covered clinical terminology, overlapping features, selected negation and uncertainty expressions, multi-site complaints, and common abbreviations.

These results support the implementation consistency of the predefined rules. They do not establish clinical sensitivity, specificity, or generalizability to unseen text expressions.

Further assessment focuses on a limited audit of real chief complaint records rather than expanding the rule vocabulary.



## 4. Focused Rule Audit

### 4.1 Audit Sample Selection

**Objective**

Select a small, reproducible sample of real ED chief complaint records to assess the behavior of the four prespecified clinical text indicators.

**Method**

A targeted sample will be constructed from the complete encounter-level dataset, including:

- All four encounters with a positive radiating leg pain indicator.
- Five encounters with an explicit sciatica indicator.
- Five encounters with an explicit LBP indicator.
- Five encounters with broad back pain but without explicit LBP.
- Six encounters containing negation, uncertainty, or potentially ambiguous text expressions.

Sampling will use a fixed random seed where applicable. Duplicate encounters across sampling groups will be removed.

The sample is purposive rather than statistically representative. Its purpose is to identify obvious implementation errors and document limitations, not to estimate population-level sensitivity or specificity.

The audit will assess agreement between the extracted indicators and observable textual evidence without consulting final ICD diagnoses or opioid dispensing outcomes.


In [13]:

# 4.1 Reproducible targeted audit sample

AUDIT_SEED = 42

def sample_group(frame, n, label):
    sampled = frame.sample(
        n=min(n, len(frame)),
        random_state=AUDIT_SEED
    ).copy()
    sampled["audit_group"] = label
    return sampled


audit_parts = []

# All radiating leg pain matches
audit_parts.append(
    sample_group(
        df_cc[df_cc["cc_radiating_leg_pain"] == 1],
        4,
        "radiation_positive"
    )
)

# Explicit sciatica matches
audit_parts.append(
    sample_group(
        df_cc[df_cc["cc_sciatica"] == 1],
        5,
        "sciatica_positive"
    )
)

# Explicit LBP matches
audit_parts.append(
    sample_group(
        df_cc[df_cc["cc_explicit_lbp"] == 1],
        5,
        "lbp_positive"
    )
)

# Broad back pain without explicit LBP
audit_parts.append(
    sample_group(
        df_cc[
            (df_cc["cc_broad_back_pain"] == 1)
            & (df_cc["cc_explicit_lbp"] == 0)
        ],
        5,
        "broad_only"
    )
)

# Negation / uncertainty / ambiguity candidates
audit_parts.append(
    sample_group(
        df_cc[
            df_cc["chiefcomplaint"].str.contains(
                r"\b(?:no|denies?|r\s*/\s*o|vs\.?|versus)\b",
                case=False,
                regex=True,
                na=False
            )
        ],
        6,
        "assertion_boundary"
    )
)

audit_sample = (
    pd.concat(audit_parts, ignore_index=True)
    .drop_duplicates(subset=["stay_id"])
    .reset_index(drop=True)
)

audit_columns = [
    "stay_id",
    "chiefcomplaint",
    *CLINICAL_FEATURES,
    "audit_group"
]

display(audit_sample[audit_columns])

print("Total unique audit records:", len(audit_sample))
print(audit_sample["audit_group"].value_counts())


,stay_id,chiefcomplaint,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain,audit_group
0,31653717,BACK PAIN RADIATING TO LEG,0,0,1,1,radiation_positive
1,36466255,LBP RAD TO RIGHT LEG,1,0,1,1,radiation_positive
2,30530210,RIGHT SIDED BACK PAIN INTO LEG,0,0,1,1,radiation_positive
3,34653865,LBP RAD TO LLE,1,0,1,1,radiation_positive
4,30134794,SCIATICA,0,1,0,0,sciatica_positive
5,37100440,SCIATICA,0,1,0,0,sciatica_positive
6,36389921,L SCIATIC PAIN,0,1,0,0,sciatica_positive
7,30306914,"Rash, SCIATICA",0,1,0,0,sciatica_positive
8,33634613,SCIATICA PAIN,0,1,0,0,sciatica_positive
9,35655220,"Constipation, L Lower back pain",1,0,1,0,lbp_positive


Total unique audit records: 25
audit_group
assertion_boundary    6
sciatica_positive     5
lbp_positive          5
broad_only            5
radiation_positive    4
Name: count, dtype: int64



### 4.2 Manual Rule Audit

**Objective**

Evaluate whether the four clinical indicators are consistent with observable textual evidence in a small sample of real ED chief complaints.

**Method**

The 25 records selected in Section 4.1 will be reviewed for apparent false-positive matches, unsupported clinical inference, and ambiguous expressions.

Because the original assertion-boundary sample primarily contained rule-out expressions unrelated to back pain or sciatica, a limited supplementary search will identify up to six real complaints containing both relevant clinical terminology and explicit negation or diagnostic uncertainty.

Review decisions will distinguish:

- **Consistent:** The extracted indicators are supported by the documented text.
- **Potential mismatch:** The rule output appears inconsistent with the prespecified feature definition.
- **Ambiguous:** The available text does not support a confident interpretation.

The audit is purposive and exploratory. It will not be used to estimate clinical sensitivity, specificity, or population-level classification accuracy.

Final ICD diagnoses and opioid dispensing outcomes will not be consulted.


In [14]:

# 4.2 Targeted review of clinically relevant assertion expressions

assertion_terms = (
    r"\b(?:no|denies?|without|r\s*/\s*o|rule\s+out|vs\.?|versus)\b"
)

clinical_terms = (
    r"\b(?:back|lbp|lumbar|lumbago|sciatic\w*)\b"
)

assertion_mask = df_cc["chiefcomplaint"].str.contains(
    assertion_terms,
    case=False,
    regex=True,
    na=False
)

clinical_mask = df_cc["chiefcomplaint"].str.contains(
    clinical_terms,
    case=False,
    regex=True,
    na=False
)

# Exclude encounters already in the initial audit sample
candidate_audit = df_cc.loc[
    assertion_mask
    & clinical_mask
    & ~df_cc["stay_id"].isin(audit_sample["stay_id"])
]

assertion_review = candidate_audit.sample(
    n=min(6, len(candidate_audit)),
    random_state=42
)

print("Relevant assertion candidates:", len(candidate_audit))
print("Supplementary records:", len(assertion_review))

display(
    assertion_review[
        ["chiefcomplaint", *CLINICAL_FEATURES]
    ]
)


Relevant assertion candidates: 7
Supplementary records: 6


,chiefcomplaint,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain
9053,CALL BACK-RULE OUT OBSTRUCTION,0,0,0,0
276127,R/O CELLULITIS TO LOWER BACK,0,0,0,0
352607,"Back pain, R/O CAUDA EQUINA",0,0,1,0
311880,R/O SCIATICA,0,0,0,0
348316,"Back pain, R/O INFECTION",0,0,1,0
329185,"Lower back pain, R/O SEPTIC JOINT",1,0,1,0



#### Interpretation

A focused qualitative review was conducted using 25 purposively selected chief complaint records and six additional records containing clinically relevant assertion or uncertainty expressions.

The initial sample covered explicit LBP, sciatica, broad back pain, radiating leg pain, and general assertion-boundary cases. The supplementary sample addressed clinically relevant rule-out expressions that were not adequately represented in the initial sample.

All 31 reviewed records were consistent with the prespecified operational feature definitions, with no obvious rule mismatches identified.

The review also highlighted that expressions such as "RIGHT SIDED BACK PAIN INTO LEG" may support a directional pain indicator without establishing a clinically confirmed radicular condition.

No further rule changes were warranted based on the reviewed examples.

This purposive audit was intended to identify obvious rule errors and document implementation limitations. It does not establish clinical sensitivity, specificity, or generalizable classification accuracy.



### 4.3 Audit Findings & Rule Freeze

**Objective**

Summarize the results of targeted regression testing and focused manual review, document remaining limitations, and freeze the clinical text feature definitions for downstream analysis.

**Audit Summary**

The lightweight clinical rule set was evaluated through two complementary quality assurance procedures:

1. **Regression testing:** All 22 prespecified test cases passed after a targeted correction to the broad back pain pattern for multi-site expressions.
2. **Focused manual review:** A total of 31 real ED chief complaint records were reviewed, comprising 25 purposively selected records and six supplementary assertion-related examples.

The manual review covered explicit LBP, sciatica, broad back pain, radiating leg pain, and selected negation or diagnostic uncertainty expressions.

No obvious mismatches with the prespecified operational feature definitions were identified among the reviewed records. No additional rule changes were considered necessary.

**Final Feature Set**

| Feature | Operational Meaning |
|---|---|
| `cc_explicit_lbp` | Explicit textual evidence of low back pain |
| `cc_sciatica` | Affirmed sciatica-related terminology |
| `cc_broad_back_pain` | Broad back pain indicator, including explicit LBP |
| `cc_radiating_leg_pain` | Explicit back-to-leg radiation expressions |
| `cc_text_missing` | No usable normalized chief complaint text |

The four clinical indicators are binary and non-mutually-exclusive. The inclusion relationship between explicit LBP and broad back pain is preserved.

**Remaining Limitations**

- The rules recognize selected clinical expressions and do not provide comprehensive coverage of all possible abbreviations, spelling errors, or symptom descriptions.
- Negation and uncertainty handling is limited to explicitly defined patterns.
- Ambiguous descriptions may not be reliably interpreted.
- The purposive audit sample is not statistically representative and does not support estimates of clinical sensitivity, specificity, or precision.
- The indicators capture documented triage language rather than clinically confirmed conditions.

**Rule Freeze Decision**

The rule definitions and implementation in `src/chief_complaint_rules.py` are frozen as the P3 v1.0 feature specification.

No further lexical expansion or rule optimization will be undertaken based on ICD diagnoses, opioid dispensing outcomes, or downstream model performance.

The resulting features are ready for encounter-level export and subsequent evaluation in P6.



## 5. Final Feature Export & P4/P6 Handoff

### 5.1 Final Feature Export

**Objective**

Export the finalized encounter-level chief complaint features using the rule definitions frozen in Section 4.

**Method**

The feature dataset will retain the ED encounter identifier, original chief complaint text, normalized text, text missingness indicator, and four prespecified clinical concept indicators.

All records from the original ED triage extraction will be retained without ICD-based cohort filtering or opioid outcome selection.

The exported dataset will contain exactly one record per `stay_id`.

**Output**

`results/03_chief_complaint/chief_complaint_features.csv`

The CSV is a derived, encounter-level analytical dataset. It will remain local and excluded from GitHub tracking.

The frozen rule implementation is maintained separately in `src/chief_complaint_rules.py`, allowing the features to be regenerated from the original extraction.


In [15]:

# 5.1 Export finalized encounter-level chief complaint features

FEATURE_OUTPUT_PATH = (
    RESULTS_DIR / "chief_complaint_features.csv"
)

export_columns = [
    "stay_id",
    "chiefcomplaint",
    "cc_text_normalized",
    "cc_text_missing",
    *CLINICAL_FEATURES,
]

# Validate required columns
assert set(export_columns).issubset(df_cc.columns)

# Create the final export table
df_cc_export = df_cc[export_columns].copy()

# Encounter-level integrity
assert df_cc_export["stay_id"].notna().all()
assert df_cc_export["stay_id"].is_unique
assert len(df_cc_export) == len(df_cc_raw)

# Validate binary indicators
binary_columns = [
    "cc_text_missing",
    *CLINICAL_FEATURES,
]

assert df_cc_export[binary_columns].notna().all().all()
assert df_cc_export[binary_columns].isin([0, 1]).all().all()

# Validate prespecified feature hierarchy
assert not (
    (df_cc_export["cc_explicit_lbp"] == 1)
    & (df_cc_export["cc_broad_back_pain"] == 0)
).any()

# Validate missing-text consistency
assert (
    df_cc_export["cc_text_missing"].eq(1)
    == df_cc_export["cc_text_normalized"].eq("")
).all()

# Export the final dataset
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_cc_export.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False,
)

print("Feature dataset exported:", FEATURE_OUTPUT_PATH)
print("Total encounters:", len(df_cc_export))
print("Total columns:", len(df_cc_export.columns))

display(df_cc_export.head(10))


Feature dataset exported: /Users/elisali/Desktop/MSAI/AI_380H_Healthcare/HRP/lbp-opioid-prediction/results/03_chief_complaint/chief_complaint_features.csv
Total encounters: 425087
Total columns: 8


,stay_id,chiefcomplaint,cc_text_normalized,cc_text_missing,cc_explicit_lbp,cc_sciatica,cc_broad_back_pain,cc_radiating_leg_pain
0,30000012,CHANGE IN MENTAL STATUS,change in mental status,0,0,0,0,0
1,30000017,"ETOH, Unable to ambulate",etoh unable to ambulate,0,0,0,0,0
2,30000038,Cough,cough,0,0,0,0,0
3,30000039,s/p Fall,s p fall,0,0,0,0,0
4,30000055,L Ear pain,l ear pain,0,0,0,0,0
5,30000094,N,n,0,0,0,0,0
6,30000112,Abnormal labs,abnormal labs,0,0,0,0,0
7,30000177,"Abd pain, Vomiting",abd pain vomiting,0,0,0,0,0
8,30000202,ALLERGIC REACTION,allergic reaction,0,0,0,0,0
9,30000204,S/P ASSAULT,s p assault,0,0,0,0,0



### 5.2 Export Integrity & Reload QA

**Objective**

Verify that the exported CSV preserves encounter identifiers, record counts, feature values, and missing-text information.

**Method**

Reload the exported dataset independently from disk and compare it with the source DataFrame.

The QA will verify:

1. Expected record count and required schema.
2. Unique, nonmissing encounter identifiers.
3. Binary feature integrity and logical hierarchy.
4. Agreement of key feature counts with the in-memory dataset.
5. Preservation of normalized text and original complaint fields.

Successful checks establish that the exported dataset is structurally suitable for P4 encounter-level integration.


In [16]:

# 5.2 Reload and validate the exported feature dataset

df_cc_check = pd.read_csv(
    FEATURE_OUTPUT_PATH,
    dtype={
        "stay_id": "string",
        "chiefcomplaint": "string",
        "cc_text_normalized": "string",
        **{col: "int8" for col in binary_columns},
    },
    keep_default_na=False,
)

# Schema and encounter integrity
assert list(df_cc_check.columns) == export_columns
assert len(df_cc_check) == len(df_cc_export)
assert df_cc_check["stay_id"].is_unique
assert df_cc_check["stay_id"].notna().all()
assert df_cc_check["stay_id"].str.strip().ne("").all()

# Verify that feature values were preserved
for col in binary_columns:
    assert df_cc_check[col].isin([0, 1]).all()
    assert df_cc_check[col].equals(
        df_cc_export[col].reset_index(drop=True)
    )

# Verify original and normalized text were preserved
for col in ["chiefcomplaint", "cc_text_normalized"]:
    assert df_cc_check[col].equals(
        df_cc_export[col].reset_index(drop=True)
    )

# Verify logical feature hierarchy
assert not (
    (df_cc_check["cc_explicit_lbp"] == 1)
    & (df_cc_check["cc_broad_back_pain"] == 0)
).any()

# QA summary
export_qa = pd.DataFrame({
    "metric": [
        "Total encounters",
        "Unique stay_id",
        "Missing normalized text",
        "Explicit LBP",
        "Sciatica",
        "Broad back pain",
        "Radiating leg pain",
    ],
    "value": [
        len(df_cc_check),
        df_cc_check["stay_id"].nunique(),
        int(df_cc_check["cc_text_missing"].sum()),
        int(df_cc_check["cc_explicit_lbp"].sum()),
        int(df_cc_check["cc_sciatica"].sum()),
        int(df_cc_check["cc_broad_back_pain"].sum()),
        int(df_cc_check["cc_radiating_leg_pain"].sum()),
    ]
})

display(export_qa)

print("Final feature export and reload QA: PASS")


,metric,value
0,Total encounters,425087
1,Unique stay_id,425087
2,Missing normalized text,1332
3,Explicit LBP,6033
4,Sciatica,20
5,Broad back pain,17101
6,Radiating leg pain,4


Final feature export and reload QA: PASS



### 5.3 P4/P6 Handoff & Conclusion

**Final Deliverables**

P3 produced a reproducible, encounter-level chief complaint feature dataset using lightweight, deterministic clinical text rules. Text normalization is implemented in Section 2 of this notebook, while the clinical indicator rules are maintained in `src/chief_complaint_rules.py`. Both are required to reproduce the exported features and should be version-controlled together.

The final feature dataset is stored at:

`results/03_chief_complaint/chief_complaint_features.csv`


The exported dataset contains 425,087 unique ED encounters and eight fields:

| Field | Description | Downstream Role |
|---|---|---|
| `stay_id` | Unique ED encounter identifier | P4 integration key |
| `chiefcomplaint` | Original triage chief complaint | Audit/reference |
| `cc_text_normalized` | Deterministically normalized text | P6 TF-IDF input |
| `cc_text_missing` | Missing or non-informative text indicator | Predictive feature |
| `cc_explicit_lbp` | Explicit low back pain terminology | Rule-based predictive feature |
| `cc_sciatica` | Affirmed sciatica terminology | Rule-based predictive feature |
| `cc_broad_back_pain` | Broad back pain terminology, including explicit LBP | Rule-based predictive feature |
| `cc_radiating_leg_pain` | Explicit back-to-leg radiation expressions | Rule-based predictive feature |

**Final Quality Assurance**

The completed workflow established:

- One unique record per `stay_id`, with no missing encounter identifiers.
- 423,755 encounters with usable normalized chief complaint text and 1,332 without usable text.
- Successful extraction of four non-mutually-exclusive clinical concept indicators.
- Preservation of the explicit-LBP-to-broad-back-pain inclusion relationship.
- Successful completion of all 22 targeted regression tests.
- Focused manual review of 31 real chief complaint records without identifying obvious mismatches with the prespecified definitions.
- Successful CSV export, independent reload, and data integrity checks.

The clinical text rules were frozen as P3 v1.0 after focused quality assessment.

**P4 — Encounter-Level Integration**

P4 will merge the exported chief complaint features with the ICD-defined C1/C2 cohorts using `stay_id`.

All ED encounters from the original extraction were retained during P3. Cohort eligibility will be determined exclusively by the finalized P2 ICD codebook and additional prespecified study eligibility criteria.

The P4 integration process will preserve one record per ED encounter and verify join coverage, missingness, and cohort membership.

Chief complaint indicators will not be used to modify C1/C2 membership.

**P6 — Predictive Modeling**

P6 will evaluate two complementary chief complaint representations in the primary C1 cohort:

- **E3a:** Structured triage features plus four rule-based clinical indicators and text missingness.
- **E3b:** Structured triage features plus TF-IDF text features, with optional TruncatedSVD for dimensionality reduction.

E3a and E3b will be evaluated against the same structured-feature LightGBM baseline (E2), using consistent patient-level train/test assignments and evaluation metrics.

TF-IDF vocabulary and IDF weights, as well as any SVD components, will be fitted exclusively on training data after patient-level partitioning.

Final ICD diagnoses and opioid dispensing outcomes will not be used as predictive inputs or for post hoc optimization of the text rules.

**Limitations**

The four clinical indicators are operational text features rather than clinically validated phenotypes. Their coverage depends on the selected lexical patterns, limited assertion handling, and the brevity of ED chief complaint documentation.

The focused manual audit was purposive and does not support population-level sensitivity, specificity, or precision estimates.

P3 did not assess predictive performance. The incremental value of the text features remains an empirical question for P6.

**Conclusion**

P3 established a lightweight, transparent, and reproducible chief complaint feature engineering workflow suitable for downstream encounter integration and predictive modeling.

The workflow preserves the separation between retrospective ICD-defined cohort identification and triage-time predictive information, while avoiding the complexity of developing an independent clinical NLP classification system.
